# 🧪 MST-GNN Pipeline Smoke Test

Test toàn bộ pipeline (~5 min) với data nhỏ. Phát hiện bugs trước khi chạy thật.
- CSI300 data, giới hạn **6 tháng** (2025-01 → 2025-06)
- Training: **3 epochs** only
- 8 phases: Data → Preprocess → Graphs → Dataset → Train → Backtest → Save → Git Push


---
## Cell 1 — Setup


In [ ]:
import torch, subprocess, os, sys, time, glob, json

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

REPO_URL = "https://github.com/quocnguyen5/mst-gnn-impl.git"
WORK_DIR = "/kaggle/working/mst-gnn-impl"
if os.path.exists(WORK_DIR):
    !cd {WORK_DIR} && git pull --rebase
else:
    !git clone {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
!git log --oneline -3
!pip install -q torch_geometric akshare vnstock jieba pyarrow fastparquet tqdm 2>&1 | tail -3

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.environ["GITHUB_TOKEN"] = token
    !git remote set-url origin https://{token}@github.com/quocnguyen5/mst-gnn-impl.git
    !git config user.email "nguyennq.app@gmail.com"
    !git config user.name "Kaggle AutoPush"
    print("✅ GitHub token OK")
except:
    os.environ["GITHUB_TOKEN"] = ""
    print("⚠️ No GitHub token")
print("\n✅ Setup done!")


---
## Cell 2 — Full Pipeline Test


In [ ]:
import time, traceback, os, sys, json, glob, torch
import numpy as np
import pandas as pd
import subprocess
import pickle

T_TOTAL = time.time()
results = {}
PASS = "✅ PASS"
FAIL = "❌ FAIL"

# ═══════════════════════════════════════════════
# Phase 1: Data Loading (from git cache, no fetch)
# ═══════════════════════════════════════════════
print("=" * 60)
print("  PHASE 1: Data Loading (from cache)")
print("=" * 60)
try:
    t = time.time()
    from config import Config
    config = Config.for_csi300()

    # Load directly from cached parquet
    price_file = os.path.join(config.data.raw_data_dir, "daily_prices_300stocks_2018-01-02_2026-06-26.parquet")
    if not os.path.exists(price_file):
        # Fallback: find any matching file
        candidates = sorted(glob.glob(os.path.join(config.data.raw_data_dir, "daily_prices_*stocks*.parquet")))
        price_file = candidates[0] if candidates else None
        assert price_file, f"No price cache in {config.data.raw_data_dir}!"

    print(f"  Loading: {os.path.basename(price_file)}")
    prices = pd.read_parquet(price_file)
    prices["date"] = pd.to_datetime(prices["date"])

    industry = pd.read_csv(os.path.join(config.data.raw_data_dir, "industry_classification.csv"))
    news_files = sorted(glob.glob(os.path.join(config.data.raw_data_dir, "news_*.parquet")))
    news = pd.read_parquet(news_files[-1]) if news_files else pd.DataFrame()
    sh_file = os.path.join(config.data.raw_data_dir, "shareholding.csv")
    shareholding = pd.read_csv(sh_file) if os.path.exists(sh_file) else pd.DataFrame()

    # ★ LIMIT to 6 months
    prices = prices[(prices["date"] >= "2025-01-01") & (prices["date"] <= "2025-06-30")]
    n_stocks = prices["stock_code"].nunique()

    assert len(prices) > 0, "No data after filter!"
    print(f"  Rows: {len(prices):,} | Stocks: {n_stocks}")
    print(f"  Range: {prices['date'].min().date()} → {prices['date'].max().date()}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 1: Data Loading"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 1: Data Loading"] = FAIL

# ═══════════════════════════════════════════════
# Phase 2: Preprocessing
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 2: Preprocessing")
print("=" * 60)
try:
    t = time.time()
    from data.preprocessing import StockPreprocessor

    preprocessor = StockPreprocessor(
        lookback_window=config.data.lookback_window,
        prediction_horizon=config.data.prediction_horizon,
    )
    processed_df, samples, trading_dates = preprocessor.process_pipeline(prices)

    # Get active stocks per date (same as run_main.py)
    active_stocks = {}
    for date in trading_dates:
        active_stocks[date] = preprocessor.get_active_stocks(processed_df, date)

    assert len(trading_dates) > 10, f"Only {len(trading_dates)} dates!"
    print(f"  Trading dates: {len(trading_dates)}")
    print(f"  Samples: {len(samples):,}")
    print(f"  Range: {trading_dates[0]} → {trading_dates[-1]}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 2: Preprocessing"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 2: Preprocessing"] = FAIL

# ═══════════════════════════════════════════════
# Phase 3: Graph Building (~2 min)
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 3: Graph Building")
print("=" * 60)
try:
    t = time.time()
    from data.graph_builder import GraphBuilder

    graph_builder = GraphBuilder(
        comovement_window=config.data.comovement_window,
        comovement_threshold=config.data.comovement_threshold,
        num_topics=config.data.num_topics,
        topic_similarity_threshold=config.data.topic_similarity_threshold,
    )

    graphs = graph_builder.build_temporal_multilayer_graphs(
        trading_dates=trading_dates,
        price_df=processed_df,
        industry_df=industry,
        shareholding_df=shareholding,
        news_df=news,
        active_stocks_per_date=active_stocks,
    )

    assert len(graphs) > 0, "No graphs built!"
    print(f"  Graphs: {len(graphs)}")
    print(f"  Time: {(time.time()-t)/60:.1f} min")
    results["Phase 3: Graph Building"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 3: Graph Building"] = FAIL

# ═══════════════════════════════════════════════
# Phase 4: Dataset Creation
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 4: Dataset Creation")
print("=" * 60)
try:
    t = time.time()
    from data.dataset import DatasetBuilder

    dataset_builder = DatasetBuilder(
        train_ratio=config.data.train_ratio,
        val_ratio=config.data.val_ratio,
        test_ratio=config.data.test_ratio,
        cache_dir=config.data.processed_data_dir,
    )
    snapshots = dataset_builder.build_snapshots(trading_dates, samples, graphs)
    train_ds, val_ds, test_ds = dataset_builder.split_dataset(snapshots)

    assert len(train_ds) > 0 and len(val_ds) > 0 and len(test_ds) > 0
    print(f"  Snapshots: {len(snapshots)}")
    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 4: Dataset Creation"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 4: Dataset Creation"] = FAIL

# ═══════════════════════════════════════════════
# Phase 5: Training (3 epochs)
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 5: Training (3 epochs)")
print("=" * 60)
try:
    t = time.time()
    from models.mst_gnn import MSTGNN
    from train import Trainer

    config.train.num_epochs = 3
    config.train.patience = 999
    config.train.checkpoint_interval = 1
    config.train.experiment_name = "smoke_test"
    os.makedirs(config.train.save_dir, exist_ok=True)

    model = MSTGNN.from_config(config)
    print(f"  Model params: {model.count_parameters():,}")

    trainer = Trainer(
        model=model, config=config,
        train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    )
    test_metrics = trainer.train(auto_resume=False)

    assert "accuracy" in test_metrics
    print(f"\n  Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  DAMRR:    {test_metrics['damrr']:.4f}")
    print(f"  Loss:     {test_metrics['loss']:.4f}")
    print(f"  Best Epoch: {trainer.best_epoch}")
    print(f"  Time: {(time.time()-t)/60:.1f} min")
    results["Phase 5: Training"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 5: Training"] = FAIL

# ═══════════════════════════════════════════════
# Phase 6: Backtest
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 6: Backtest")
print("=" * 60)
try:
    t = time.time()
    from backtest import TradingSimulator

    dates, codes, preds, scores, returns = trainer.get_predictions(test_ds)
    simulator = TradingSimulator(
        top_k_stocks=config.backtest.top_k_stocks,
        transaction_cost=config.backtest.transaction_cost,
    )
    backtest_results = simulator.simulate(dates, codes, scores, returns)
    report = simulator.generate_report(backtest_results)
    print(report.to_string(index=False))

    save_path = os.path.join(config.train.save_dir, "cumulative_returns_smoke_test.png")
    simulator.plot_results(backtest_results, dates=dates, save_path=save_path)
    print(f"  Chart: {save_path}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 6: Backtest"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 6: Backtest"] = FAIL

# ═══════════════════════════════════════════════
# Phase 7: Persistence
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 7: Persistence")
print("=" * 60)
try:
    # Check checkpoint
    best_ckpt = "checkpoints/best_model_smoke_test.pt"
    if os.path.exists(best_ckpt):
        ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        has_hist = "train_history" in ckpt and len(ckpt["train_history"]) > 0
        print(f"  Checkpoint: ✅ epoch {ckpt['epoch']}, history={has_hist}")
    else:
        print(f"  Checkpoint: ❌ NOT FOUND")
        for f in glob.glob("checkpoints/*.pt"):
            print(f"    Found: {os.path.basename(f)}")

    # Save JSON
    results_json = {
        "dataset": "smoke_test",
        "test_metrics": {k: round(v, 4) for k, v in test_metrics.items()},
        "best_epoch": trainer.best_epoch,
        "train_history": trainer.train_history,
        "val_history": trainer.val_history,
    }
    json_path = "checkpoints/results_smoke_test.json"
    with open(json_path, "w") as f:
        json.dump(results_json, f, indent=2)
    # Verify
    with open(json_path) as f:
        loaded = json.load(f)
    assert "test_metrics" in loaded
    print(f"  JSON: ✅ saved & verified")
    results["Phase 7: Persistence"] = PASS
except Exception as e:
    print(f"  ERROR: {e}"); traceback.print_exc()
    results["Phase 7: Persistence"] = FAIL

# ═══════════════════════════════════════════════
# Phase 8: Git Push
# ═══════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 8: Git Push")
print("=" * 60)
try:
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        def _run(cmd):
            r = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR, timeout=120)
            return r.returncode, r.stdout + r.stderr
        _run(["git", "add", "-A"])
        rc, out = _run(["git", "commit", "-m", "Smoke test results"])
        if rc != 0 and "nothing to commit" in out:
            print("  Nothing to commit")
        else:
            rc, _ = _run(["git", "push"])
            if rc != 0:
                _run(["git", "pull", "--rebase", "--no-edit"])
                rc, _ = _run(["git", "push"])
            if rc != 0:
                rc, _ = _run(["git", "push", "--force"])
            print(f"  {'✅ Push OK' if rc == 0 else '❌ Push failed'}")
        results["Phase 8: Git Push"] = PASS
    else:
        print("  ⏭️ Skipped (no token)")
        results["Phase 8: Git Push"] = "⏭️ SKIP"
except Exception as e:
    print(f"  ERROR: {e}")
    results["Phase 8: Git Push"] = FAIL

# ═══════════════════════════════════════════════
# REPORT
# ═══════════════════════════════════════════════
total_min = (time.time() - T_TOTAL) / 60
print("\n\n" + "═" * 60)
print("  🧪 SMOKE TEST REPORT")
print("═" * 60)
all_pass = True
for phase, status in results.items():
    print(f"  {status}  {phase}")
    if "FAIL" in status:
        all_pass = False
print(f"\n  Total time: {total_min:.1f} min")
if all_pass:
    print("\n  🎉 ALL PASSED — Safe to run full experiments!")
else:
    print("\n  ⚠️  FAILED — Fix issues above first!")
print("═" * 60)


---
## Cell 3 — Backtest Chart


In [ ]:
from IPython.display import Image as IPImage, display
chart = "checkpoints/cumulative_returns_smoke_test.png"
if os.path.exists(chart):
    display(IPImage(filename=chart))
else:
    print("No chart — check Phase 6.")
